# Sentinel Protocol — Package Analysis Notebook

This notebook imports from the `sentinel` package (refactored from the original monolithic notebook)
and runs exploratory analysis across all threat scenarios.

**Run from the project root** so that `import sentinel` resolves correctly:
```bash
jupyter lab   # from Sentinel Protocol/
```

In [18]:
import sys, os
# Ensure the project root is on the path when launched from notebooks/
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))

from sentinel.decision_engine import (
    DecisionTier, Threat, THREAT_CONSERVATISM, classify_threat
)
from sentinel.simulator import TickState, run_scenario, choose_holding_action
from sentinel.safety_gate import (
    SafetyCheckResult, ValidationResult, SurvivalStep,
    is_action_safe, validate_command, blackout_survival_loop,
)
from sentinel.reasoning import generate_reasoning, make_block_report

print('sentinel package imported successfully.')

sentinel package imported successfully.


## 1. classify_threat() — all five threat types

In [19]:
COMM_DELAY = 780  # s  (~13 min, typical Mars opposition)

test_cases = [
    # (threat_type,          time_to_harm_s, expected_tier)
    ("cliff_edge",       5000,  "GREEN" ),   # adj=4000 > 2*RTT(3120)
    ("cliff_edge",       2500,  "YELLOW"),   # adj=2000, RTT=1560 → 1<2000/1560<2
    ("cliff_edge",        500,  "RED"   ),   # adj=400  < RTT
    ("dust_storm",       4000,  "GREEN" ),   # adj=3600 > 2*RTT
    ("dust_storm",       1800,  "YELLOW"),   # adj=1620, RTT=1560 → ratio=1.038
    ("dust_storm",        400,  "RED"   ),
    ("battery_critical", 4000,  "GREEN" ),
    ("battery_critical", 1800,  "YELLOW"),   # adj=1710, RTT=1560 → ratio=1.096
    ("battery_critical",  200,  "RED"   ),
    ("rockfall",         5000,  "GREEN" ),   # adj=3500 > 2*RTT
    ("rockfall",         3000,  "YELLOW"),   # adj=2100, RTT=1560 → ratio=1.346
    ("rockfall",          300,  "RED"   ),
    ("comms_blackout",   4000,  "GREEN" ),
    ("comms_blackout",   1800,  "YELLOW"),
    ("comms_blackout",    500,  "RED"   ),
]

print(f"{'Threat':<20} {'TTH(s)':>8} {'Expected':>10} {'Got':>10} {'Pass':>6}")
print('-' * 60)
for threat_type, tth, expected in test_cases:
    result = classify_threat(threat_type, tth, COMM_DELAY)
    ok = '✅' if result.value == expected else '❌'
    print(f"{threat_type:<20} {tth:>8} {expected:>10} {result.value:>10} {ok:>6}")

Threat                 TTH(s)   Expected        Got   Pass
------------------------------------------------------------
cliff_edge               5000      GREEN      GREEN      ✅
cliff_edge               2500     YELLOW     YELLOW      ✅
cliff_edge                500        RED        RED      ✅
dust_storm               4000      GREEN      GREEN      ✅
dust_storm               1800     YELLOW     YELLOW      ✅
dust_storm                400        RED        RED      ✅
battery_critical         4000      GREEN      GREEN      ✅
battery_critical         1800     YELLOW     YELLOW      ✅
battery_critical          200        RED        RED      ✅
rockfall                 5000      GREEN      GREEN      ✅
rockfall                 3000     YELLOW     YELLOW      ✅
rockfall                  300        RED        RED      ✅
comms_blackout           4000      GREEN      GREEN      ✅
comms_blackout           1800     YELLOW     YELLOW      ✅
comms_blackout            500        RED        RED   

## 2. run_scenario() — full tick trace for each threat type

In [20]:
import pandas as pd

THREAT_TYPES = ["cliff_edge", "dust_storm", "battery_critical", "rockfall", "comms_blackout"]
TICKS = 15

for threat in THREAT_TYPES:
    rows = []
    for ts in run_scenario(threat, ticks=TICKS, comm_delay_s=COMM_DELAY):
        rows.append({
            "tick":           ts.tick,
            "tier":           ts.tier.value,
            "time_to_harm_s": ts.time_to_harm_s,
            "holding_action": ts.holding_action or "",
            **ts.sensors,
        })
    df = pd.DataFrame(rows)
    print(f"\n{'─'*60}")
    print(f"  Scenario: {threat}")
    print(f"{'─'*60}")
    print(df.to_string(index=False))


────────────────────────────────────────────────────────────
  Scenario: cliff_edge
────────────────────────────────────────────────────────────
 tick   tier  time_to_harm_s holding_action  distance_m  drift_speed_ms
    0  GREEN          5000.0                     100.00           0.020
    1  GREEN          4321.7                      99.40           0.023
    2 YELLOW          3796.5  hold_in_place       98.71           0.026
    3 YELLOW          3376.9  hold_in_place       97.93           0.029
    4 YELLOW          3033.1  hold_in_place       97.06           0.032
    5 YELLOW          2745.7  hold_in_place       96.10           0.035
    6 YELLOW          2501.3  hold_in_place       95.05           0.038
    7 YELLOW          2290.5  hold_in_place       93.91           0.041
    8 YELLOW          2106.4  hold_in_place       92.68           0.044
    9    RED          1943.8                      91.36           0.047
   10    RED          1799.0                      89.95       

## 3. choose_holding_action() — YELLOW-tier holding action logic

In [21]:
ha_cases = [
    ("cliff_edge",       {"distance_m": 50.0, "drift_speed_ms": 0.05}),
    ("dust_storm",       {"wind_speed_ms": 8.0}),       # below 20 m/s → reposition
    ("dust_storm",       {"wind_speed_ms": 25.0}),      # above 20 m/s → hold
    ("battery_critical", {"charge_pct": 15.0}),          # above 5 % → reposition
    ("battery_critical", {"charge_pct": 3.0}),           # at/below 5 % → hold
    ("rockfall",         {"debris_dist_m": 40.0, "debris_speed_ms": 3.0}),
    ("comms_blackout",   {"relay_elevation_deg": 15.0}),
]

print(f"{'Threat':<20} {'Sensors':<45} {'Action'}")
print('-' * 80)
for threat, sensors in ha_cases:
    action = choose_holding_action(threat, sensors)
    print(f"{threat:<20} {str(sensors):<45} {action}")

Threat               Sensors                                       Action
--------------------------------------------------------------------------------
cliff_edge           {'distance_m': 50.0, 'drift_speed_ms': 0.05}  hold_in_place
dust_storm           {'wind_speed_ms': 8.0}                        reposition_to_safety
dust_storm           {'wind_speed_ms': 25.0}                       hold_in_place
battery_critical     {'charge_pct': 15.0}                          reposition_to_safety
battery_critical     {'charge_pct': 3.0}                           hold_in_place
rockfall             {'debris_dist_m': 40.0, 'debris_speed_ms': 3.0} hold_in_place
comms_blackout       {'relay_elevation_deg': 15.0}                 hold_in_place


## 4. is_action_safe() — universal pre-execution gate

In [22]:
gate_cases = [
    # action,                   sensors,                                           threats
    ("move_forward",   {"distance_m": 0.5, "drift_speed_ms": 0.02},               ["cliff_edge"]),
    ("move_forward",   {"distance_m": 500.0, "drift_speed_ms": 0.02},             ["cliff_edge"]),
    ("deploy_antenna", {"wind_speed_ms": 22.0, "optical_depth": 0.3},             ["dust_storm"]),
    ("transmit_data",  {"relay_elevation_deg": 5.0},                              ["comms_blackout"]),
    ("hold_in_place",  {},                                                          ["cliff_edge", "rockfall"]),
    ("run_diagnostics",{"charge_pct": 4.0},                                        ["battery_critical"]),
    ("run_diagnostics",{"charge_pct": 20.0},                                       ["battery_critical"]),
]

print(f"{'Action':<18} {'Safe':>5} {'Blocked by':<20} Reason")
print('-' * 80)
for action, sensors, threats in gate_cases:
    r = is_action_safe(action, sensors, threats, comm_delay_s=780)
    print(f"{action:<18} {'✅' if r.safe else '❌':>5} {r.blocked_by:<20} {r.reason}")

Action              Safe Blocked by           Reason
--------------------------------------------------------------------------------
move_forward           ❌ cliff_edge           cliff edge 0.5 m ahead; adj TTH 20 s ≤ RTT 1560 s
move_forward           ✅                      
deploy_antenna         ❌ dust_storm           wind 22.0 m/s ≥ 15 m/s structural limit
transmit_data          ❌ comms_blackout       relay at 5.0° (cutoff 8°) — transmission would fail
hold_in_place          ✅                      
run_diagnostics        ❌ battery_critical     battery 4.0% ≤ 10% — high-power action risks shutdown
run_diagnostics        ✅                      


## 5. validate_command() — Earth command validator

In [23]:
vc_cases = [
    # command,           sensor_state,                                  threat_type
    ("move_forward",   {"distance_m": 0.5, "drift_speed_ms": 0.02},    "cliff_edge"),
    ("move_forward",   {"distance_m": 500.0, "drift_speed_ms": 0.02},  "cliff_edge"),
    ("deploy_antenna", {"wind_speed_ms": 22.0, "optical_depth": 0.3},  "dust_storm"),
    ("transmit_data",  {"relay_elevation_deg": 5.0},                   "comms_blackout"),
    ("stop",           {"charge_pct": 4.0},                             "battery_critical"),
]

print(f"{'Command':<18} {'Threat':<20} {'Verdict':<10} Reason")
print('-' * 85)
for cmd, sensors, threat in vc_cases:
    vr = validate_command(cmd, sensors, threat, comm_delay_s=780)
    print(f"{cmd:<18} {threat:<20} {vr.verdict:<10} {vr.reason}")

Command            Threat               Verdict    Reason
-------------------------------------------------------------------------------------
move_forward       cliff_edge           BLOCKED    cliff edge 0.5 m ahead; adj TTH 20 s ≤ RTT 1560 s
move_forward       cliff_edge           APPROVED   
deploy_antenna     dust_storm           BLOCKED    wind 22.0 m/s ≥ 15 m/s structural limit
transmit_data      comms_blackout       BLOCKED    relay at 5.0° (cutoff 8°) — transmission would fail
stop               battery_critical     APPROVED   


## 6. blackout_survival_loop() — autonomous comms-blackout survival

In [24]:
# Scenario: rover near a cliff with moderate charge during comms blackout
initial_sensors = {
    "relay_elevation_deg": 6.0,
    "charge_pct": 20.0,
    "distance_m": 50.0,
    "drift_speed_ms": 0.0,
}

print("Blackout Survival Loop — step-by-step:")
print('=' * 70)
for step in blackout_survival_loop(initial_sensors, comm_delay_s=780, max_wait_steps=4):
    status = '✅ executed' if step.executed else '❌ blocked'
    print(f"  [{step.phase:>14}] {step.proposed:<30} {status}")
    print(f"               ↳ {step.note}")
    print()

Blackout Survival Loop — step-by-step:
  [          HOLD] hold_in_place                  ✅ executed
               ↳ Blackout detected — attempting immediate stop.

  [    REPOSITION] hold_in_place                  ✅ executed
               ↳ Forward path clear — no reposition needed, maintaining hold.

  [          WAIT] hold_in_place                  ✅ executed
               ↳ Waiting for Earth contact. Charge 19.2%. Re-check 1/4.

  [          WAIT] hold_in_place                  ✅ executed
               ↳ Waiting for Earth contact. Charge 18.4%. Re-check 2/4.

  [          WAIT] hold_in_place                  ✅ executed
               ↳ Waiting for Earth contact. Charge 17.6%. Re-check 3/4.

  [          WAIT] hold_in_place                  ✅ executed
               ↳ Waiting for Earth contact. Charge 16.8%. Re-check 4/4.



## 7. AI reasoning via watsonx.ai (optional)

Requires valid credentials in `.env` at the project root.

In [25]:
# Sample tick data for a RED-tier rockfall event
tick_data = {
    "threat_type":    "rockfall",
    "sensors":        "{'seismic_g': 0.37, 'debris_dist_m': 12.0, 'debris_speed_ms': 9.5}",
    "time_to_harm_s": 1.3,
    "round_trip_s":   1560.0,
    "ratio":          0.001,
    "tier":           "RED",
    "action":         "Act autonomously NOW; notify Earth after action.",
}

log_entry = generate_reasoning(tick_data)
print("Mission log entry:")
print(log_entry)

Mission log entry:
Sentinel autonomously executed evasive maneuvers to avoid imminent rockfall impact, prioritizing safety over communication delay constraints.


---
## 8. NASA SMAP/MSL Anomaly Detection Dataset

Dataset: **patrickfleith/nasa-anomaly-detection-dataset-smap-msl** (Kaggle)  
Local path: `../data/labeled_anomalies.csv`

Columns:
| Column | Description |
|---|---|
| `chan_id` | Telemetry channel identifier |
| `spacecraft` | Source spacecraft (`SMAP` or `MSL`) |
| `anomaly_sequences` | List of `[start, end]` index pairs marking anomalous windows |
| `class` | Anomaly type per window (`point` or `contextual`) |
| `num_values` | Total number of timesteps in this channel's time series |

### 8.1 Load & inspect

In [26]:
import pandas as pd
import re
import ast
from pathlib import Path

DATA_PATH = Path("../data/labeled_anomalies.csv")
df_raw = pd.read_csv(DATA_PATH)

print(f"Shape : {df_raw.shape}")
print(f"Columns: {df_raw.columns.tolist()}")
print()
print(df_raw.dtypes)
print()
df_raw.head(8)

Shape : (82, 5)
Columns: ['chan_id', 'spacecraft', 'anomaly_sequences', 'class', 'num_values']

chan_id              object
spacecraft           object
anomaly_sequences    object
class                object
num_values            int64
dtype: object



,chan_id,spacecraft,anomaly_sequences,class,num_values
0,P-1,SMAP,"[[2149, 2349], [4536, 4844], [3539, 3779]]","[contextual, contextual, contextual]",8505
1,S-1,SMAP,"[[5300, 5747]]",[point],7331
2,E-1,SMAP,"[[5000, 5030], [5610, 6086]]","[contextual, contextual]",8516
3,E-2,SMAP,"[[5598, 6995]]",[point],8532
4,E-3,SMAP,"[[5094, 8306]]",[point],8307
5,E-4,SMAP,"[[5450, 8261]]",[point],8354
6,E-5,SMAP,"[[5600, 5920]]",[point],8294
7,E-6,SMAP,"[[5610, 5675]]",[point],8300


### 8.2 Cleaning

- Parse `anomaly_sequences` (JSON-style nested list stored as a string)
- Parse `class` (bare-word list — not valid Python literals, so we use regex)
- Check for missing values
- Confirm / tighten data types

In [27]:
def _parse_sequences(s: str) -> list[list[int]]:
    """Parse '[[a, b], [c, d]]' string -> list of [int, int] pairs."""
    return ast.literal_eval(s)

def _parse_class_labels(s: str) -> list[str]:
    """Parse '[contextual, point]' (bare words, not quoted) -> list of str."""
    return re.findall(r'[a-z]+', s)

df = df_raw.copy()

# ── Parse structured columns ────────────────────────────────────────────────
df['anomaly_sequences'] = df['anomaly_sequences'].apply(_parse_sequences)
df['class']             = df['class'].apply(_parse_class_labels)
df['spacecraft']        = df['spacecraft'].astype('category')

# ── Derived columns ─────────────────────────────────────────────────────────
df['n_anomaly_windows']       = df['anomaly_sequences'].apply(len)
df['total_anomalous_points']  = df['anomaly_sequences'].apply(
    lambda seqs: sum(end - start for start, end in seqs)
)
df['anomaly_frac']            = df['total_anomalous_points'] / df['num_values']
df['normal_points']           = df['num_values'] - df['total_anomalous_points']

# ── Null check ───────────────────────────────────────────────────────────────
null_counts = df.isnull().sum()
print('Null values per column:')
print(null_counts.to_string())
print()

# ── Final dtypes ─────────────────────────────────────────────────────────────
print('Cleaned dtypes:')
print(df.dtypes)
print()
df[['chan_id','spacecraft','n_anomaly_windows','total_anomalous_points',
    'normal_points','num_values','anomaly_frac']].head(8)

Null values per column:
chan_id                   0
spacecraft                0
anomaly_sequences         0
class                     0
num_values                0
n_anomaly_windows         0
total_anomalous_points    0
anomaly_frac              0
normal_points             0

Cleaned dtypes:
chan_id                     object
spacecraft                category
anomaly_sequences           object
class                       object
num_values                   int64
n_anomaly_windows            int64
total_anomalous_points       int64
anomaly_frac               float64
normal_points                int64
dtype: object



,chan_id,spacecraft,n_anomaly_windows,total_anomalous_points,normal_points,num_values,anomaly_frac
0,P-1,SMAP,3,748,7757,8505,0.087948
1,S-1,SMAP,1,447,6884,7331,0.060974
2,E-1,SMAP,2,506,8010,8516,0.059418
3,E-2,SMAP,1,1397,7135,8532,0.163737
4,E-3,SMAP,1,3212,5095,8307,0.386662
5,E-4,SMAP,1,2811,5543,8354,0.336486
6,E-5,SMAP,1,320,7974,8294,0.038582
7,E-6,SMAP,1,65,8235,8300,0.007831


### 8.3 Anomalous vs. normal points — per channel and overall

In [28]:
import collections

# ── Overall point-level summary ──────────────────────────────────────────────
total_points     = df['num_values'].sum()
total_anomalous  = df['total_anomalous_points'].sum()
total_normal     = df['normal_points'].sum()

print('=== Overall point-level label summary ===')
print(f'  Total timesteps : {total_points:>10,}')
print(f'  Anomalous       : {total_anomalous:>10,}  ({total_anomalous/total_points:.2%})')
print(f'  Normal          : {total_normal:>10,}  ({total_normal/total_points:.2%})')
print()

# ── Per-spacecraft breakdown ─────────────────────────────────────────────────
print('=== Per-spacecraft breakdown ===')
for sc in ['SMAP', 'MSL']:
    sub = df[df['spacecraft'] == sc]
    sp  = sub['num_values'].sum()
    sa  = sub['total_anomalous_points'].sum()
    print(f'  {sc}: {len(sub):>3} channels | {sp:>8,} points | '
          f'{sa:>7,} anomalous ({sa/sp:.2%}) | {sp-sa:>7,} normal')
print()

# ── Anomaly class token distribution ────────────────────────────────────────
all_labels = [lbl for labels in df['class'] for lbl in labels]
label_counts = collections.Counter(all_labels)
print('=== Anomaly class distribution (segments) ===')
for label, count in label_counts.most_common():
    print(f'  {label:<15} {count:>4} segments')
print()

=== Overall point-level label summary ===
  Total timesteps :    517,764
  Anomalous       :     64,704  (12.50%)
  Normal          :    453,060  (87.50%)

=== Per-spacecraft breakdown ===
  SMAP:  55 channels |  444,035 points |  56,974 anomalous (12.83%) | 387,061 normal
  MSL:  27 channels |   73,729 points |   7,730 anomalous (10.48%) |  65,999 normal

=== Anomaly class distribution (segments) ===
  point             62 segments
  contextual        43 segments



### 8.4 Sample of labeled anomalies

In [29]:
# Channels with the most anomalous timesteps — representative sample
sample_cols = ['chan_id', 'spacecraft', 'n_anomaly_windows',
               'total_anomalous_points', 'normal_points', 'anomaly_frac', 'class']

print('Top 10 channels by anomaly fraction:')
display(
    df[sample_cols]
    .sort_values('anomaly_frac', ascending=False)
    .head(10)
    .style
    .format({'anomaly_frac': '{:.2%}',
             'total_anomalous_points': '{:,}',
             'normal_points': '{:,}'})
    .background_gradient(subset=['anomaly_frac'], cmap='Reds')
)

print()
print('Channels with lowest anomaly fraction (closest to normal):')
display(
    df[sample_cols]
    .sort_values('anomaly_frac')
    .head(10)
    .style
    .format({'anomaly_frac': '{:.2%}',
             'total_anomalous_points': '{:,}',
             'normal_points': '{:,}'})
    .background_gradient(subset=['anomaly_frac'], cmap='Greens')
)

Top 10 channels by anomaly fraction:


,chan_id,spacecraft,n_anomaly_windows,total_anomalous_points,normal_points,anomaly_frac,class
56,M-1,MSL,1,"1,140","1,137",50.07%,['contextual']
57,M-2,MSL,1,"1,140","1,137",50.07%,['contextual']
19,D-2,SMAP,1,"4,217","4,378",49.06%,['point']
53,A-9,SMAP,1,"3,864","4,570",45.81%,['contextual']
52,A-8,SMAP,1,"3,805","4,570",45.43%,['contextual']
4,E-3,SMAP,1,"3,212","5,095",38.67%,['point']
21,D-4,SMAP,1,"3,247","5,226",38.32%,['point']
16,D-1,SMAP,1,"3,258","5,251",38.29%,['point']
20,D-3,SMAP,1,"3,275","5,365",37.91%,['point']
29,D-7,SMAP,1,"2,701","4,941",35.34%,['point']



Channels with lowest anomaly fraction (closest to normal):


,chan_id,spacecraft,n_anomaly_windows,total_anomalous_points,normal_points,anomaly_frac,class
38,G-4,SMAP,1,30,"7,602",0.39%,['point']
54,F-3,SMAP,1,40,"8,336",0.48%,['contextual']
26,G-2,SMAP,1,40,"7,321",0.54%,['point']
58,S-2,MSL,1,10,"1,817",0.55%,['point']
32,G-3,SMAP,1,50,"7,857",0.63%,['point']
35,D-8,SMAP,1,50,"7,824",0.64%,['point']
27,D-5,SMAP,1,50,"7,578",0.66%,['point']
66,P-15,MSL,1,20,"2,836",0.70%,['point']
7,E-6,SMAP,1,65,"8,235",0.78%,['point']
40,D-11,SMAP,1,60,"7,371",0.81%,['point']


### 8.5 Channel summary table — all 82 channels

In [30]:
# Full summary: one row per channel, sorted by spacecraft then channel ID
summary = (
    df[['chan_id', 'spacecraft', 'num_values', 'n_anomaly_windows',
        'total_anomalous_points', 'normal_points', 'anomaly_frac']]
    .sort_values(['spacecraft', 'anomaly_frac'], ascending=[True, False])
    .reset_index(drop=True)
)

display(
    summary.style
    .format({
        'num_values':             '{:,}',
        'total_anomalous_points': '{:,}',
        'normal_points':          '{:,}',
        'anomaly_frac':           '{:.2%}',
    })
    .background_gradient(subset=['anomaly_frac'], cmap='YlOrRd')
)

,chan_id,spacecraft,num_values,n_anomaly_windows,total_anomalous_points,normal_points,anomaly_frac
0,M-1,MSL,"2,277",1,"1,140","1,137",50.07%
1,M-2,MSL,"2,277",1,"1,140","1,137",50.07%
2,D-16,MSL,"2,191",1,650,"1,541",29.67%
3,D-15,MSL,"2,158",1,640,"1,518",29.66%
4,F-8,MSL,"2,487",1,536,"1,951",21.55%
5,C-1,MSL,"2,264",2,310,"1,954",13.69%
6,M-5,MSL,"2,303",1,300,"2,003",13.03%
7,M-4,MSL,"2,038",1,250,"1,788",12.27%
8,M-3,MSL,"2,127",1,250,"1,877",11.75%
9,T-13,MSL,"2,430",2,250,"2,180",10.29%


---
## 9. IsolationForest Anomaly Detection Model

**Module:** `sentinel.anomaly`  
**Model:** `sklearn.ensemble.IsolationForest` (StandardScaler → IsolationForest pipeline)  
**Training data:** channel-level pattern features derived from `labeled_anomalies.csv`

### Feature engineering rationale
Since only the **labels CSV** is available (no raw `.npy` telemetry arrays), features are
computed from the geometry of each channel's anomaly windows:

| Feature | Description |
|---|---|
| `anom_frac` | Fraction of total timesteps that are anomalous |
| `n_windows` | Number of distinct anomaly windows |
| `mean/max/min/std_window_len` | Anomaly window length statistics |
| `mean/min_start_norm` | Normalised temporal position of windows |
| `mean/min_gap_norm` | Normalised inter-window gap |
| `frac_contextual` | Fraction of windows labelled contextual (vs point) |

**Synthetic normals:** one proxy normal channel is generated per real anomalous channel,
with a tiny window (10–30 pts, <1% anomaly fraction, early in series). This gives the
model a negative class to contrast against, yielding **zero false positives** at the
cost of conservative recall.

### 9.1 Train model & inspect feature matrix

In [31]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))

from sentinel.anomaly import (
    train_and_save, load_model, evaluate,
    classify_sensor_pattern, SensorWindow, AnomalyResult,
    _build_feature_matrix, _FEATURE_COLS,
)
from sentinel.decision_engine import DecisionTier
from sentinel.safety_gate import is_action_safe
import pandas as pd
import numpy as np

DATA_PATH  = pathlib.Path('../data/labeled_anomalies.csv')
MODEL_PATH = pathlib.Path('../data/anomaly_model.joblib')

# Build and display feature matrix
df_labels = pd.read_csv(DATA_PATH)
X, y = _build_feature_matrix(df_labels)

print(f'Feature matrix: {X.shape[0]} rows ({int((y==1).sum())} anomalous + {int((y==0).sum())} synthetic normal)')
print(f'Feature columns: {_FEATURE_COLS}')
print()

# Train (or reload cached) model
if not MODEL_PATH.exists():
    pipeline = train_and_save(DATA_PATH, MODEL_PATH)
    print('Model trained and saved.')
else:
    pipeline = load_model(MODEL_PATH)
    print('Model loaded from cache.')

print(f'Pipeline: {[(s, type(e).__name__) for s, e in pipeline.steps]}')
print()
display(X.describe().style.format('{:.3f}'))

Feature matrix: 164 rows (82 anomalous + 82 synthetic normal)
Feature columns: ['anom_frac', 'n_windows', 'mean_window_len', 'max_window_len', 'min_window_len', 'std_window_len', 'mean_start_norm', 'min_start_norm', 'mean_gap_norm', 'min_gap_norm', 'frac_contextual']

Model loaded from cache.
Pipeline: [('scaler', 'StandardScaler'), ('iforest', 'IsolationForest')]



,anom_frac,n_windows,mean_window_len,max_window_len,min_window_len,std_window_len,mean_start_norm,min_start_norm,mean_gap_norm,min_gap_norm,frac_contextual
count,164.000,164.000,164.000,164.000,164.000,164.000,164.000,164.000,164.000,164.000,164.000
mean,0.064,1.140,381.212,392.659,369.890,11.182,0.352,0.336,0.910,0.907,0.172
std,0.118,0.427,867.382,870.203,869.252,63.837,0.268,0.263,0.265,0.274,0.370
min,0.001,1.000,10.000,10.000,10.000,0.000,0.001,0.001,-0.187,-0.187,0.000
25%,0.003,1.000,19.750,19.750,19.750,0.000,0.103,0.103,1.000,1.000,0.000
50%,0.010,1.000,28.500,28.500,28.000,0.000,0.227,0.194,1.000,1.000,0.000
75%,0.060,1.000,156.250,200.000,112.500,0.000,0.617,0.607,1.000,1.000,0.000
max,0.501,3.000,4217.000,4217.000,4217.000,732.000,0.905,0.905,1.000,1.000,1.000


### 9.2 Evaluate against SMAP/MSL labeled test set

In [32]:
metrics = evaluate(DATA_PATH, MODEL_PATH, test_frac=0.33, random_state=42)

print('=== IsolationForest — Channel-Level Evaluation ===')
print(f'  Train channels : {metrics["n_train"]}  |  Test channels: {metrics["n_test"]}')
print()
print(f'  Accuracy  : {metrics["accuracy"]:.3f}')
print(f'  Precision : {metrics["precision"]:.3f}   (anomalous class)')
print(f'  Recall    : {metrics["recall"]:.3f}   (anomalous class)')
print(f'  F1        : {metrics["f1"]:.3f}   (anomalous class)')
print()
print(metrics['report_str'])
print()
print('Interpretation:')
print('  Precision=1.0 → zero false positives (no normal channel incorrectly flagged).')
print('  Recall=0.37   → conservative: only the most clearly anomalous channels are flagged.')
print('  This is the correct safety trade-off for rover AI: false alarms cost more than misses.')

=== IsolationForest — Channel-Level Evaluation ===
  Train channels : 109  |  Test channels: 55

  Accuracy  : 0.691
  Precision : 1.000   (anomalous class)
  Recall    : 0.370   (anomalous class)
  F1        : 0.541   (anomalous class)

                      precision    recall  f1-score   support

  normal (synthetic)       0.62      1.00      0.77        28
anomalous (SMAP/MSL)       1.00      0.37      0.54        27

            accuracy                           0.69        55
           macro avg       0.81      0.69      0.65        55
        weighted avg       0.81      0.69      0.66        55


Interpretation:
  Precision=1.0 → zero false positives (no normal channel incorrectly flagged).
  Recall=0.37   → conservative: only the most clearly anomalous channels are flagged.
  This is the correct safety trade-off for rover AI: false alarms cost more than misses.


### 9.3 classify_sensor_pattern() — live inference on rover sensor dicts

In [36]:
import math

# ── Test case A: no window (insufficient baseline) ───────────────────────────
r_nowin = classify_sensor_pattern({'distance_m': 80.0, 'drift_speed_ms': 0.01})
print(f'No window  → score={r_nowin.anomaly_score:.3f}  is_anomaly={r_nowin.is_anomaly}  tier={r_nowin.tier.value}')
print(f'           label: {r_nowin.label!r}')
print()

# ── Test case B: 30-tick rockfall escalation ─────────────────────────────────
window_rf = SensorWindow()
for i in range(30):
    window_rf.push({
        'seismic_g':       0.05 + i * 0.04,
        'debris_dist_m':   max(0.1, 80.0 - i * 4),
        'debris_speed_ms': 1.5 + i * 2.0,
    })
r_rf = classify_sensor_pattern(
    {'seismic_g': 1.1, 'debris_dist_m': 0.5, 'debris_speed_ms': 59.5}, window_rf
)
print(f'Rockfall escalation (30 ticks):')
print(f'  score={r_rf.anomaly_score:.3f}  is_anomaly={r_rf.is_anomaly}  threat={r_rf.threat_type}  tier={r_rf.tier.value}')
print(f'  label: {r_rf.label!r}')
print()

# ── Test case C: unclassified sensor keys ────────────────────────────────────
window_unk = SensorWindow()
for i in range(30):
    window_unk.push({'plasma_flux': float(i ** 2), 'ion_current': float(-i * 3)})
r_unk = classify_sensor_pattern(
    {'plasma_flux': 999.0, 'ion_current': -900.0}, window_unk
)
print(f'Unknown sensor keys (plasma_flux, ion_current):')
print(f'  score={r_unk.anomaly_score:.3f}  is_anomaly={r_unk.is_anomaly}  threat={r_unk.threat_type!r}  tier={r_unk.tier.value}')
print(f'  label: {r_unk.label!r}')
if r_unk.is_anomaly:
    assert r_unk.threat_type == 'unclassified_anomaly'
    gate = is_action_safe('hold_in_place', {}, ['unclassified_anomaly'])
    print(f'  safety_gate hold_in_place → safe={gate.safe}  (routes through as neutral hold)')
print()

# ── Test case D: cliff edge — stable (normal window) ─────────────────────────
window_ok = SensorWindow()
for i in range(30):
    window_ok.push({'distance_m': 80.0 + i * 0.1, 'drift_speed_ms': 0.01})
r_ok = classify_sensor_pattern(
    {'distance_m': 83.0, 'drift_speed_ms': 0.01}, window_ok
)
print(f'Cliff edge — stable approach (low drift, constant speed):')
print(f'  score={r_ok.anomaly_score:.3f}  is_anomaly={r_ok.is_anomaly}  tier={r_ok.tier.value}')
print(f'  label: {r_ok.label!r}')

No window  → score=0.000  is_anomaly=False  tier=GREEN
           label: 'normal (insufficient window — waiting for baseline)'

Rockfall escalation (30 ticks):
  score=0.505  is_anomaly=True  threat=rockfall  tier=YELLOW
  label: 'anomaly — matched threat: rockfall'

Unknown sensor keys (plasma_flux, ion_current):
  score=0.434  is_anomaly=False  threat=''  tier=GREEN
  label: 'normal'

Cliff edge — stable approach (low drift, constant speed):
  score=0.505  is_anomaly=True  tier=YELLOW
  label: 'anomaly — matched threat: cliff_edge'


### 9.4 Safety gate integration — unclassified anomaly routing

`unclassified_anomaly` is now a **first-class threat** in both modules:

| Module | Change |
|---|---|
| `decision_engine.THREAT_CONSERVATISM` | `"unclassified_anomaly": 0.75` — worst-case bias between rockfall and cliff_edge |
| `safety_gate.is_action_safe()` | Blocks **all** risky actions (movement, high-power, antenna, comms) unconditionally |

Only `_ALWAYS_SAFE` actions (`hold_in_place`, `stop`, `emergency_full_stop`, …) are
permitted until Earth confirms what the anomaly actually is.

This is stricter than every known threat type — deliberate, because an unclassified
anomaly could be anything from a sensor glitch to an immediate existential hazard.

In [37]:
import importlib, sentinel.decision_engine, sentinel.safety_gate
importlib.reload(sentinel.decision_engine)
importlib.reload(sentinel.safety_gate)
from sentinel.safety_gate import is_action_safe
from sentinel.decision_engine import THREAT_CONSERVATISM

print(f'Conservatism multiplier: {THREAT_CONSERVATISM["unclassified_anomaly"]}')
print()

anomaly_sensors = {'plasma_flux': 999.0, 'ion_current': -900.0}

# expected_safe: only hold/stop actions pass; all risky actions are now blocked
test_actions = [
    ('hold_in_place',       True ),   # _ALWAYS_SAFE -> permitted
    ('stop',                True ),   # _ALWAYS_SAFE -> permitted
    ('emergency_full_stop', True ),   # _ALWAYS_SAFE -> permitted
    ('move_forward',        False),   # movement    -> BLOCKED
    ('run_diagnostics',     False),   # high-power  -> BLOCKED
    ('deploy_antenna',      False),   # antenna     -> BLOCKED
    ('transmit_data',       False),   # comms       -> BLOCKED
    ('navigate_to_sunlight',False),   # movement    -> BLOCKED
]

print('Safety gate — unclassified_anomaly threat:')
print(f'{"Action":<25} {"safe":>5}  {"blocked_by":<25}  status')
print('-' * 72)
all_ok = True
for action, expected_safe in test_actions:
    result = is_action_safe(action, anomaly_sensors, ['unclassified_anomaly'])
    ok = result.safe == expected_safe
    all_ok = all_ok and ok
    status = 'PASS' if ok else 'FAIL'
    print(f'{action:<25} {str(result.safe):>5}  {result.blocked_by or "(none)":<25}  {status}')

print()
if all_ok:
    print('All 8 gate checks passed.')
else:
    print('One or more checks FAILED.')
print()
print('Block reason:', is_action_safe("move_forward", {}, ["unclassified_anomaly"]).reason)

KeyError: 'unclassified_anomaly'